Count orders and revenue

In [1]:
import sys
from confluent_kafka import Consumer

In [2]:
crash_after = 0 # never crash

In [5]:
consumer = Consumer({"bootstrap.servers": "localhost:9092",
                     "group.id": "order-counter",
                     "auto.offset.reset": "earliest",
                     "enable.auto.commit": False,
                     "session.timeout.ms": 6000})
consumer.subscribe(["orders"])

In [7]:
%run order.ipynb

In [8]:
seen, revenue, idle = 0, 0.0, 0
while idle < 5:
    msg = consumer.poll(1.0)
    
    if msg is None:
        if consumer.assignment():
            idle += 1
        continue
    if msg.error():
        print("error: ", msg.error())
        continue
    idle = 0
    order = decode(msg.value())
    seen += 1
    revenue += order["qty"] * order["price"]
    
    if seen == crash_after:
        print(f"CRASH after {seen} orders, before the next commit.")
        sys.exit(1)
    
    if seen % 50 == 0:
        consumer.commit(asynchronous=False)
        print(f"processed {seen} orders, offsets committed")

if seen % 50:
    consumer.commit(asynchronous=False)
consumer.close()
print(f"done: {seen} orders, revenue {revenue:,.2f}")


done: 2 orders, revenue 30.00


In [ ]:
# Run this
# papermill consumer.ipynb output_consumer.ipynb